In [1]:
!pip install tensorflow pandas scikit-learn scikit-fuzzy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 17.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

df = pd.DataFrame(X, columns=cancer.feature_names)
df['target'] = y

print("Dataset features:")
print(df.columns)
print("\nFirst 5 rows of the dataset:")
print(df.head())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nData split into {X_train_scaled.shape[0]} training samples and {X_test_scaled.shape[0]} testing samples.")

Dataset features:
Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='object')

First 5 rows of the dataset:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00  

In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

print("\nTraining the Deep Learning model...")
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=0)
print("Training complete.")

print("\nEvaluating model performance...")
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,537 (6.00 KB)

 Trainable params: 1,537 (6.00 KB)

 Non-trainable params: 0 (0.00 B)


Training the Deep Learning model...
Training complete.

Evaluating model performance...
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9558 - loss: 0.1054
Test Accuracy: 96.49%


In [5]:
radius_col_idx = list(cancer.feature_names).index('mean radius')
texture_col_idx = list(cancer.feature_names).index('mean texture')


scaled_radius = ctrl.Antecedent(np.arange(-3, 3, 0.1), 'scaled_radius')
scaled_texture = ctrl.Antecedent(np.arange(-3, 3, 0.1), 'scaled_texture')
prognosis = ctrl.Consequent(np.arange(0, 11, 1), 'prognosis')

scaled_radius['Small'] = fuzz.trimf(scaled_radius.universe, [-3, -3, 0])
scaled_radius['Medium'] = fuzz.trimf(scaled_radius.universe, [-1.5, 0, 1.5])
scaled_radius['Large'] = fuzz.trimf(scaled_radius.universe, [0, 3, 3])

scaled_texture['Smooth'] = fuzz.trimf(scaled_texture.universe, [-3, -3, 0])
scaled_texture['Normal'] = fuzz.trimf(scaled_texture.universe, [-1.5, 0, 1.5])
scaled_texture['Coarse'] = fuzz.trimf(scaled_texture.universe, [0, 3, 3])

prognosis['Good'] = fuzz.trimf(prognosis.universe, [0, 0, 5])
prognosis['Moderate'] = fuzz.trimf(prognosis.universe, [2, 5, 8])
prognosis['Poor'] = fuzz.trimf(prognosis.universe, [5, 10, 10])


rule1 = ctrl.Rule(scaled_radius['Large'] | scaled_texture['Coarse'], prognosis['Poor'])
rule2 = ctrl.Rule(scaled_radius['Medium'] & scaled_texture['Normal'], prognosis['Moderate'])
rule3 = ctrl.Rule(scaled_radius['Small'] & scaled_texture['Smooth'], prognosis['Good'])
rule4 = ctrl.Rule(scaled_radius['Large'] & scaled_texture['Normal'], prognosis['Poor'])


prognosis_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4])
prognosis_simulation = ctrl.ControlSystemSimulation(prognosis_ctrl)

print("\nFuzzy Inference System created successfully.")


Fuzzy Inference System created successfully.


In [8]:
def predict_and_interpret(patient_data_scaled, feature_names):
    """
    Takes a single sample of scaled patient data, predicts with DL,
    and interprets with FIS.
    """
    patient_data_reshaped = np.expand_dims(patient_data_scaled, axis=0)

    dl_prediction_prob = model.predict(patient_data_reshaped, verbose=0)[0][0]
    dl_prediction = 'Malignant (High Risk)' if dl_prediction_prob > 0.5 else 'Benign (Low Risk)'


    radius_idx = list(feature_names).index('mean radius')
    texture_idx = list(feature_names).index('mean texture')

    prognosis_simulation.input['scaled_radius'] = patient_data_scaled[radius_idx]
    prognosis_simulation.input['scaled_texture'] = patient_data_scaled[texture_idx]
    prognosis_simulation.compute()

    prognosis_score = prognosis_simulation.output['prognosis']

    prognosis_level = "Unknown"
    if prognosis_score <= 5:
        prognosis_level = "Good"
    elif 5 < prognosis_score <= 8:
        prognosis_level = "Moderate"
    else:
        prognosis_level = "Poor"

    print("--- Integrated Prognosis Report ---")
    print(f"Deep Learning Prediction: {dl_prediction} (Probability: {dl_prediction_prob:.4f})")
    print(f"Fuzzy System Prognosis Score: {prognosis_score:.2f}")

    print("\nFuzzy Interpretation:")
    print(f"Based on a 'mean radius' value of {patient_data_scaled[radius_idx]:.2f} (scaled) and "
          f"a 'mean texture' value of {patient_data_scaled[texture_idx]:.2f} (scaled), "
          f"the fuzzy system suggests the prognosis is '{prognosis_level}'.")
    print("-----------------------------------")


sample_index = 10
test_sample = X_test_scaled[sample_index]
true_label = y_test[sample_index]
true_diagnosis = 'Malignant' if true_label == 1 else 'Benign'

print(f"\n--- Running analysis for sample #{sample_index} ---")
print(f"True Diagnosis: {true_diagnosis}")

predict_and_interpret(test_sample, cancer.feature_names)


--- Running analysis for sample #10 ---
True Diagnosis: Malignant
--- Integrated Prognosis Report ---
Deep Learning Prediction: Malignant (High Risk) (Probability: 0.9115)
Fuzzy System Prognosis Score: 7.66

Fuzzy Interpretation:
Based on a 'mean radius' value of -0.26 (scaled) and a 'mean texture' value of 1.42 (scaled), the fuzzy system suggests the prognosis is 'Moderate'.
-----------------------------------
